# Twisted Bilayer Graphene (TBG) Hamiltonian

This notebook builds a full TBG tight-binding Hamiltonian as an MPO in four steps:

1. **Two honeycomb layer MPOs** — NN hopping for each layer in the quantics binary
   representation using `kineticintra2DNNhex` + `kineticNNN` from `src/2D_lattice.jl`.
2. **Layer extension via direct product** — prepend the 1-core projectors
   $(1\pm\sigma_z)/2$ to extend each $L$-qubit honeycomb MPO to an $(L+1)$-qubit
   MPO in which the MSB selects the layer.
3. **Interlayer hopping via QTCI** — define an exponentially decaying hopping
   $T(\mathbf{r}_i^{(1)}, \mathbf{r}_j^{(2)}) = t_\perp e^{-\alpha|\Delta\mathbf{r}|}$
   between the two (relatively twisted) real-space lattices, learn it with
   `hopping2MPO` from `src/Hamiltonian.jl`, and compress into an MPO on the
   same extended site space.
4. **Assembly** — $H_\text{TBG} = H_{L1} + H_{L2} + H_\text{inter}$ with a
   Hermiticity validation via the full matrix representation.

**Encoding convention** ($(L+1)$ qubits, $2N$ sites total):
$$
  \underbrace{b_1}_{\text{layer}} \underbrace{b_2 \cdots b_{L+1}}_{\text{position}}
  \quad
  b_1 = 0 \;\Rightarrow\; \text{layer 1}, \qquad
  b_1 = 1 \;\Rightarrow\; \text{layer 2}
$$


In [1]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
include("../src/TensorBinding.jl")
using .TensorBinding
using QuanticsTCI
import TensorCrossInterpolation as TCI

---
## 1. Two honeycomb lattice MPOs

Each honeycomb layer lives on $N = 2^L$ sites encoded in $L$ qubits.
We use $L_x + L_y = L$ with $N_x = 2^{L_x}$ sites per row and $N_y = 2^{L_y}$ rows.

The honeycomb Hamiltonian decomposes into:
- **Intra-row** bonds: `kineticintra2DNNhex` (NN with a checkerboard A/B sublattice mask)
- **Inter-row** bonds: `kineticNNN(..., Nx)` (vertical hops spanning $N_x$ sites)

Both layers share the same `honey_sites` (the $L$ position qubits), because the
twist angle only affects the **interlayer** hopping — not the intra-layer geometry.
The layer qubit `ext_sites[1]` is allocated now so that the extended space is
consistent throughout.


In [2]:
# ── System parameters ──────────────────────────────────────────────────────────
Lx  = 4          # qubits for x-direction:  Nx = 2^Lx = 4 sites per row
Ly  = 3          # qubits for y-direction:  Ny = 2^Ly = 2 rows
L   = Lx + Ly    # L = 3 position qubits per layer
N   = 2^L        # N = 8 sites per layer
N_ext = 2N        # 2N sites in the extended (bilayer) space
L_ext = L + 1    # L+1 qubits in the extended space
t   = 1.0        # intra-layer NN hopping amplitude

# ── Extended site space ────────────────────────────────────────────────────────
# ext_sites[1]     = layer qubit  (MSB: 0 → layer 1, 1 → layer 2)
# ext_sites[2:L+1] = L position qubits (shared by both layers)
ext_sites   = siteinds("Qubit", L + 1)
honey_sites = ext_sites[2:end]       # L-site space for a single layer
xvals       = 0:N-1

# Diagonal MPO encoding the uniform NN hopping amplitude on position qubits
hops = TensorBinding.qtt_mpo(L, xvals, honey_sites, _ -> t)

# ── Layer 1 honeycomb MPO ──────────────────────────────────────────────────────
H_honey1 = +(
    TensorBinding.kineticintra2DNNhex(Lx, Ly, honey_sites, hops, 1),
    TensorBinding.kineticNNN(L, honey_sites, hops, 2^Lx);
    cutoff = 1e-8
)
println("H_honey1 : L=$(length(H_honey1))  maxlinkdim=$(ITensorMPS.maxlinkdim(H_honey1))")

# ── Layer 2 honeycomb MPO (same geometry; twist only affects interlayer terms) ─
H_honey2 = +(
    TensorBinding.kineticintra2DNNhex(Lx, Ly, honey_sites, hops, 1),
    TensorBinding.kineticNNN(L, honey_sites, hops, 2^Lx);
    cutoff = 1e-8
)
println("H_honey2 : L=$(length(H_honey2))  maxlinkdim=$(ITensorMPS.maxlinkdim(H_honey2))")

H_honey1 : L=7  maxlinkdim=5
H_honey2 : L=7  maxlinkdim=5


---
## 2. Intralayer Hamiltonians via direct product with 1-core $(1\pm\sigma_z)/2$

We extend each $L$-qubit honeycomb MPO to the $(L+1)$-qubit bilayer space by
prepending a **single-site (1-core) layer projector**:
$$
  H_{L1} = \frac{1+\sigma_z}{2} \otimes H_{\text{honey1}}, \qquad
  H_{L2} = \frac{1-\sigma_z}{2} \otimes H_{\text{honey2}}
$$
where $\frac{1+\sigma_z}{2} = |0\rangle\langle 0|$ (layer 1 projector) and
$\frac{1-\sigma_z}{2} = |1\rangle\langle 1|$ (layer 2 projector).

The 1-core is a rank-1 (bond-dimension 1) tensor $P$ acting on the layer qubit.
It is prepended to the MPO chain by:
1. Allocating a new dim-1 link `bond0` between $P$ and `H_honey[1]`.
2. Multiplying `H_honey[1]` with a scalar delta tensor to attach `bond0` as its
   left index (outer product, no contraction).
3. Assembling `[P, H_honey[1]_ext, H_honey[2], …]` into an $(L+1)$-site MPO.


In [3]:
"""
    prepend_projector(H_mpo, layer_s, layer_up) -> MPO

Extend H_mpo (on L sites) to an (L+1)-site MPO by prepending the 1-core
layer projector:
  - layer_up = true  →  P₊ = (1 + σ_z)/2 = |0⟩⟨0|  (layer 1)
  - layer_up = false →  P₋ = (1 − σ_z)/2 = |1⟩⟨1|  (layer 2)
"""
function prepend_projector(H_mpo::MPO, layer_s::Index, layer_up::Bool)
    Lh    = length(H_mpo)
    bond0 = Index(1, "Link,l=0")
    proj  = layer_up ? 1 : 2
    P     = ITensor(layer_s', layer_s, bond0)
    P[layer_s' => proj, layer_s => proj, bond0 => 1] = 1.0
    delta0 = ITensor(bond0);  delta0[bond0 => 1] = 1.0
    H1_ext = H_mpo[1] * delta0
    ext = MPO(Lh + 1);  ext[1] = P;  ext[2] = H1_ext
    for k in 3:Lh+1;  ext[k] = H_mpo[k-1];  end
    return ext
end

"""
    prepend_sigma(H_mpo, layer_s, sigma_plus) -> MPO

Extend H_mpo (on L sites) to an (L+1)-site MPO by prepending the 1-core
off-diagonal layer operator:
  - sigma_plus = true  →  σ⁺ = |0⟩⟨1|  (layer 2 → layer 1 hop)
  - sigma_plus = false →  σ⁻ = |1⟩⟨0|  (layer 1 → layer 2 hop)

Used to build the interlayer coupling H_inter = σ⁺ ⊗ V + σ⁻ ⊗ Vᵀ.
"""
function prepend_sigma(H_mpo::MPO, layer_s::Index, sigma_plus::Bool)
    Lh    = length(H_mpo)
    bond0 = Index(1, "Link,l=0")
    S     = ITensor(layer_s', layer_s, bond0)
    if sigma_plus
        S[layer_s' => 1, layer_s => 2, bond0 => 1] = 1.0   # |0⟩⟨1|
    else
        S[layer_s' => 2, layer_s => 1, bond0 => 1] = 1.0   # |1⟩⟨0|
    end
    delta0 = ITensor(bond0);  delta0[bond0 => 1] = 1.0
    H1_ext = H_mpo[1] * delta0
    ext = MPO(Lh + 1);  ext[1] = S;  ext[2] = H1_ext
    for k in 3:Lh+1;  ext[k] = H_mpo[k-1];  end
    return ext
end

layer_s = ext_sites[1]     # MSB qubit = layer index

# H_L1 = (1+σ_z)/2 ⊗ H_honey1  ── layer-1 intralayer Hamiltonian in extended space
H_L1 = prepend_projector(H_honey1, layer_s, true)
println("H_L1 : L=$(length(H_L1))  maxlinkdim=$(ITensorMPS.maxlinkdim(H_L1))")

# H_L2 = (1−σ_z)/2 ⊗ H_honey2  ── layer-2 intralayer Hamiltonian in extended space
H_L2 = prepend_projector(H_honey2, layer_s, false)
println("H_L2 : L=$(length(H_L2))  maxlinkdim=$(ITensorMPS.maxlinkdim(H_L2))")

H_L1 : L=8  maxlinkdim=5
H_L2 : L=8  maxlinkdim=5


---
## 3. Interlayer coupling V via qtt_mpo-style 1D QTCI + σ⁺/σ⁻ cores

We avoid `hopping2MPO` (which uses the 2D `DiscretizedGrid` path) and replicate
the exact pattern of `qtt_mpo`:

1. **Encode** the 2D pair $(i,j)\in[1,N]^2$ as an **interleaved** 2L-bit index
   $k\in[0,N^2-1]$.  With the bit layout $[\ldots, a_1, b_1, a_0, b_0]$ (LSB first),
   `custom_mpo` pairs sites $(2s{-}1,\,2s)$ → MPO site $s$, so the bra gets row-bits
   and the ket gets col-bits — exactly the convention used by `hopping2MPO`.

2. **1D QTCI**: call `quanticscrossinterpolate(ComplexF64, f, 0:N²-1)` (the same
   call as in `qtt_mpo`) to get a 2L-site tensor train.

3. **custom_mpo**: pair the 2L MPS sites into an L-site MPO on `honey_sites`.

4. **Kronecker product**: use `prepend_sigma` to build
   $$H_\text{inter} = \sigma^+\otimes V + \sigma^-\otimes V^T$$
   The Hermitian conjugate is explicit for real $V$: $(σ^+\otimes V)^\dagger = σ^-\otimes V^T$.


In [4]:
# ── Physical parameters ───────────────────────────────────────────────────────
θ_deg    = 0.1     # twist angle in degrees
t_inter  = 0.3
α_decay  = 1/1.5

# ── Geometry: honeycomb positions for each layer ──────────────────────────────
rs1 = TensorBinding.honeycomb_positions(L)

θ = θ_deg * π / 180
R = [cos(θ) -sin(θ); sin(θ) cos(θ)]
rs2 = (R * rs1')'

# ── V(i,j) = t_⊥ exp(−α |r1[i] − r2[j]|) on the N×N single-layer space ─────
V_f(i, j)  = t_inter * exp(-α_decay * norm(rs1[Int(i), :] - rs2[Int(j), :]))
Vt_f(i, j) = V_f(j, i)   # V^T

V_mpo  = TensorBinding.hopping2MPO(V_f,  N, honey_sites;
             tol=1e-6, type=Float64)
Vt_mpo = TensorBinding.hopping2MPO(Vt_f, N, honey_sites;
             tol=1e-6, type=Float64)

println("V_mpo  : L=$(length(V_mpo))  maxlinkdim=$(ITensorMPS.maxlinkdim(V_mpo))")
println("Vt_mpo : L=$(length(Vt_mpo)) maxlinkdim=$(ITensorMPS.maxlinkdim(Vt_mpo))")

# ── Lift to extended spacwe: H_inter = σ⁺ ⊗ V + σ⁻ ⊗ Vᵀ ──────────────────────
H_sp    = prepend_sigma(V_mpo,  layer_s, true)    # σ⁺ ⊗ V
H_sm    = prepend_sigma(Vt_mpo, layer_s, false)   # σ⁻ ⊗ Vᵀ
H_inter = +(H_sp, H_sm; cutoff=1e-8)

println("H_inter : L=$(length(H_inter)) maxlinkdim=$(ITensorMPS.maxlinkdim(H_inter))")

MPS COMPUTED!
Turned into MPO!
MPS COMPUTED!
Turned into MPO!
V_mpo  : L=7  maxlinkdim=15
Vt_mpo : L=7 maxlinkdim=15
H_inter : L=8 maxlinkdim=19


---
## 4. Full TBG Hamiltonian and DoS via KPM

$$H_\text{TBG} = H_{L1} + H_{L2} + H_\text{inter}$$

All three MPOs live on the same `ext_sites` ($(L+1)$ qubits) so
ITensorMPS can add them directly.

The density of states is computed via the **Kernel Polynomial Method** (KPM):

$$\text{DoS}(\omega) = \mathrm{Tr}[\delta(\omega - H_\text{TBG})]$$

1. Build `Tn_list` — the Chebyshev polynomial MPOs $T_n(H/\text{scale})$ up to order `Ncheb`.
2. For each $\omega$, assemble the spectral MPO $A(\omega)$ with Jackson damping via `get_ldos_w_from_Tn`.
3. $\text{DoS}(\omega)$ = `real(tr(A_mpo))` — the same MPO trace used for `Tr(ρ)` in density-matrix calculations.
4. Sweep over twist angle $\theta$ and display the DoS evolution as a heatmap.


In [5]:
# ── Assemble the full TBG Hamiltonian ─────────────────────────────────────────
H_TBG = +(+(H_L1, H_L2; cutoff=1e-8), H_inter; cutoff=1e-8)
ITensorMPS.truncate!(H_TBG; maxdim=200, cutoff=1e-5)
println("H_TBG : L=$(length(H_TBG))  maxlinkdim=$(ITensorMPS.maxlinkdim(H_TBG))")

H_TBG : L=8  maxlinkdim=11


In [ ]:
# ── KPM parameters ─────────────────────────────────────────────────────────────
scale_tbg  = 50.0      # spectral bound: honeycomb bandwidth ≤ 3t, interlayer ≤ t_inter
Ncheb      = 100
maxdim_kpm = 200

# Build Chebyshev polynomial list from H_TBG once
Tn_tbg, _, scale_tbg  = TensorBinding.KPM_Tn(H_TBG, Ncheb, ext_sites; maxdim=maxdim_kpm)

# ── DoS = Tr(δ(ω-H)) ─────────────────────────────────────────────────────────
ω_phys = range(-scale_tbg, scale_tbg; length=100)
dos    = Float64[]

for ω in ω_phys
    ω_r = ω / scale_tbg
    if abs(ω_r) >= 1.0
        push!(dos, 0.0)
        continue
    end
    A_mpo = TensorBinding.get_ldos_w_from_Tn(Tn_tbg, Ncheb, ω_r; maxdim=maxdim_kpm)
    push!(dos, real(tr(A_mpo)))
end

plot(ω_phys, dos;
     xlabel  = "Energy (t)",
     ylabel  = "DoS",
     title   = "TBG DoS  (θ=$(θ_deg)°,  t⊥=$(t_inter),  N=$(N_ext) sites)",
     legend  = false,
     lw      = 2)

KPM_Tn: estimating spectral bounds via DMRG…
  E_min = -2.9346,  E_max = 5.4603
  center = 1.2628,  scale = 4.6172
43
56
61
79
85
99
106
112
118
120
123
125
125
126
127
127
127
128
127
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128
128


In [ ]:
# ── DoS vs twist-angle heatmap ───────────────────────────────────────────────
θ_sweep   = range(0.0, 5.0; length=8)
ω_sweep   = range(-3.5, 3.5; length=150)
dos_sweep = Matrix{Float64}(undef, length(ω_sweep), length(θ_sweep))

for (idx, θ_d) in enumerate(θ_sweep)
    θ_loc   = θ_d * π / 180
    R_loc   = [cos(θ_loc) -sin(θ_loc); sin(θ_loc) cos(θ_loc)]
    rs2_loc = (R_loc * rs1')'

    V_loc_f(i, j)  = t_inter * exp(-α_decay * norm(rs1[Int(i), :] - rs2_loc[Int(j), :]))
    Vt_loc_f(i, j) = V_loc_f(j, i)

    V_loc  = TensorBinding.hopping2MPO(V_loc_f,  N, honey_sites;
                 tol=1e-5, type=Float64, unfoldingscheme=:interleaved)
    Vt_loc = TensorBinding.hopping2MPO(Vt_loc_f, N, honey_sites;
                 tol=1e-5, type=Float64, unfoldingscheme=:interleaved)

    H_loc  = +(prepend_sigma(V_loc,  layer_s, true),
               prepend_sigma(Vt_loc, layer_s, false); cutoff=1e-8)
    H_full = +(+(H_L1, H_L2; cutoff=1e-8), H_loc; cutoff=1e-8)

    Tn_loc = TensorBinding.KPM_Tn(H_full / scale_tbg, Ncheb, ext_sites; maxdim=maxdim_kpm)

    for (iω, ω) in enumerate(ω_sweep)
        ω_r = ω / scale_tbg
        if abs(ω_r) >= 1.0
            dos_sweep[iω, idx] = 0.0
        else
            A_mpo = TensorBinding.get_ldos_w_from_Tn(Tn_loc, Ncheb, ω_r; maxdim=maxdim_kpm)
            dos_sweep[iω, idx] = real(tr(A_mpo))
        end
    end
    println("θ = $(round(θ_d; digits=2))° done")
end

heatmap(collect(θ_sweep), collect(ω_sweep), dos_sweep;
        xlabel         = "Twist angle θ (degrees)",
        ylabel         = "Energy (t)",
        title          = "TBG DoS vs twist angle  (N=$(N_ext) sites, t⊥=$(t_inter))",
        color          = :inferno,
        colorbar_title = "DoS")
